# 2차 — 클래스 불균형 처리 재검증 (2단계, 멀티시드)

## 배경
예전(난임결과 예측 해커톤1)에 단일 시드(1004)로 한 번만 봤음:
- `scale_pos_weight`(계산값 ≈2.87) → 약 -0.0003
- `is_unbalance=True` → 약 -0.0003

근데 이건 노이즈 표준편차(0.00011, 10시드 측정값)의 약 2.7배라 "진짜 손해"인지 "운"인지
단일 시드로는 확신하기 어려움. 이번엔:
1. **1단계 탐색**: 더 넓은 그리드(여러 scale_pos_weight 값 + SMOTE + 언더샘플링)를 3시드로 빠르게 스캔
2. **2단계 정밀검증**: 1단계에서 baseline보다 분명히 나은 것만 10시드 배깅으로 확정 (기존 벤치마크 0.74036/0.74037과 직접 비교 가능)

## 누수 방지
- 리샘플링(SMOTE/언더샘플링)은 **각 fold의 train 부분에만** 적용. validation fold는 원본 그대로 평가
  (리샘플링은 y를 보고 데이터를 합성/삭제하는 과정이라, val에 적용하면 그 자체로 누수)
- 이 노트북은 test를 전혀 안 씀 (순수 CV 검증용) — 인코딩 누수 걱정 자체가 없음

## 사전 준비
`pip install imbalanced-learn --break-system-packages` 또는 (uv 쓰시니) `uv add imbalanced-learn`
설치 필요 (SMOTENC, RandomUnderSampler).

---
**저장 위치:** `experiment_history/2차/2차_class_imbalance_recheck.ipynb`


In [3]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
import warnings

warnings.filterwarnings("ignore")


In [4]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEEDS_STAGE1 = [1004, 42, 7]                                     # 빠른 탐색용
SEEDS_STAGE2 = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]    # 정밀검증용 (기존 벤치마크와 동일 시드셋)

# stage1에서 baseline보다 이만큼 이상 좋아야 stage2로 넘어갈 후보로 인정 (느슨한 기준 — 3시드라 노이즈 큼)
STAGE2_PROMOTION_THRESHOLD = 0.0002


## 1. 데이터 로드 + 전처리 (train만 — 이 실험은 test 안 씀)

In [5]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))  # train만 사용 (test 자체를 안 씀)

y = df_train[TARGET_COL].values.astype(int)
X = df_train.drop(columns=[TARGET_COL])
cat_col_idx = [X.columns.get_loc(c) for c in cat_cols]  # SMOTENC용 범주형 컬럼 인덱스

pos, neg = int(y.sum()), len(y) - int(y.sum())
base_spw = neg / pos
print(f"양성={pos} ({pos/len(y):.2%}), 음성={neg} ({neg/len(y):.2%})")
print(f"계산된 scale_pos_weight(neg/pos) = {base_spw:.4f}")
print(f"train shape: {X.shape}")


양성=66228 (25.83%), 음성=190123 (74.17%)
계산된 scale_pos_weight(neg/pos) = 2.8707
train shape: (256351, 67)


## 2. 시작 파라미터 로드

In [6]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")
print(json.dumps(base_params, indent=2, ensure_ascii=False))


파라미터 출처: ../lgbm_robust_best_params.json
{
  "n_estimators": 1225,
  "learning_rate": 0.03735839746471122,
  "num_leaves": 253,
  "max_depth": 5,
  "min_child_samples": 41,
  "subsample": 0.699825092698368,
  "colsample_bytree": 0.7084506083750325,
  "reg_alpha": 3.9529019873195606,
  "reg_lambda": 5.86734589907409,
  "min_split_gain": 0.5142456412311088
}


## 3. 평가 함수 (리샘플링은 train fold에만 적용)

In [7]:
def evaluate_config(seeds, lgbm_overrides=None, resample_fn=None):
    """resample_fn(X_tr, y_tr) -> (X_tr', y_tr'): fold의 train 부분에만 적용.
    val은 절대 리샘플링하지 않음 (resample_fn=None이면 아무 처리 안 함, 즉 baseline).
    반환: (시드별 AUC 배열, 시드별 OOF 배열, 멀티시드 배깅 OOF AUC)"""
    extra = lgbm_overrides or {}
    seed_aucs = []
    oof_per_seed = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        oof = np.zeros(len(y))
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, y_tr = X.iloc[tr_idx].copy(), y[tr_idx].copy()
            X_val = X.iloc[val_idx]

            if resample_fn is not None:
                X_tr, y_tr = resample_fn(X_tr, y_tr)

            m = LGBMClassifier(**base_params, random_state=seed, verbose=-1, **extra)
            m.fit(X_tr, y_tr)
            oof[val_idx] = m.predict_proba(X_val)[:, 1]

        seed_aucs.append(roc_auc_score(y, oof))
        oof_per_seed.append(oof)

    oof_per_seed = np.array(oof_per_seed)
    bagged_auc = roc_auc_score(y, oof_per_seed.mean(axis=0))
    return np.array(seed_aucs), oof_per_seed, bagged_auc


def make_smote_resampler():
    def fn(X_tr, y_tr):
        sm = SMOTENC(categorical_features=cat_col_idx, random_state=0, sampling_strategy=1.0)
        return sm.fit_resample(X_tr, y_tr)
    return fn


def make_undersample_resampler():
    def fn(X_tr, y_tr):
        rus = RandomUnderSampler(random_state=0, sampling_strategy=1.0)
        return rus.fit_resample(X_tr, y_tr)
    return fn


## 4. Stage 1 — 3시드 빠른 탐색

In [8]:
configs = [
    ("baseline", {}, None),
    ("spw=1.5", {"scale_pos_weight": 1.5}, None),
    ("spw=2.0", {"scale_pos_weight": 2.0}, None),
    (f"spw=계산값({base_spw:.2f})", {"scale_pos_weight": base_spw}, None),
    ("spw=3.5", {"scale_pos_weight": 3.5}, None),
    ("spw=4.5", {"scale_pos_weight": 4.5}, None),
    ("is_unbalance=True", {"is_unbalance": True}, None),
    ("SMOTE 오버샘플링(1:1)", {}, make_smote_resampler()),
    ("RandomUnderSampler(1:1)", {}, make_undersample_resampler()),
]

print(f"Stage 1 탐색 (시드 {len(SEEDS_STAGE1)}개: {SEEDS_STAGE1})\n")
print(f"{'설정':<28} | {'평균AUC':>8} | {'표준편차':>8} | {'베이스라인差':>12} | 시간")
print("-" * 75)

stage1_results = {}
baseline_mean = None
for label, overrides, resample_fn in configs:
    t0 = time.time()
    aucs, _, _ = evaluate_config(SEEDS_STAGE1, overrides, resample_fn)
    mean_auc, std_auc = aucs.mean(), aucs.std()
    if label == "baseline":
        baseline_mean = mean_auc
    diff = mean_auc - baseline_mean if baseline_mean is not None else 0.0
    stage1_results[label] = {"overrides": overrides, "resample_fn": resample_fn, "mean": mean_auc, "std": std_auc}
    print(f"{label:<28} | {mean_auc:>8.5f} | {std_auc:>8.5f} | {diff:>+12.5f} | {time.time()-t0:>4.0f}s")


Stage 1 탐색 (시드 3개: [1004, 42, 7])

설정                           |    평균AUC |     표준편차 |       베이스라인差 | 시간
---------------------------------------------------------------------------
baseline                     |  0.73991 |  0.00005 |     +0.00000 |   52s
spw=1.5                      |  0.73985 |  0.00007 |     -0.00006 |   51s
spw=2.0                      |  0.73983 |  0.00010 |     -0.00008 |   55s
spw=계산값(2.87)                |  0.73973 |  0.00006 |     -0.00018 |   57s
spw=3.5                      |  0.73964 |  0.00005 |     -0.00027 |   60s
spw=4.5                      |  0.73959 |  0.00012 |     -0.00033 |   63s
is_unbalance=True            |  0.73972 |  0.00005 |     -0.00019 |   57s
SMOTE 오버샘플링(1:1)             |  0.71222 |  0.00012 |     -0.02769 | 1062s
RandomUnderSampler(1:1)      |  0.73915 |  0.00009 |     -0.00076 |   33s


## 5. Stage 2 — baseline보다 분명히 나은 것만 10시드 정밀검증

In [9]:
candidates = [
    label for label, r in stage1_results.items()
    if label != "baseline" and (r["mean"] - baseline_mean) > STAGE2_PROMOTION_THRESHOLD
]
print(f"Stage 2 정밀검증 후보 (baseline보다 +{STAGE2_PROMOTION_THRESHOLD} 이상): "
      f"{candidates if candidates else '없음'}\n")

print(f"{'설정':<28} | {'10시드 평균':>10} | {'표준편차':>8} | {'10시드 배깅AUC':>13}")
print("-" * 70)

stage2_results = {}
for label in ["baseline"] + candidates:
    r = stage1_results[label]
    aucs2, _, bagged_auc2 = evaluate_config(SEEDS_STAGE2, r["overrides"], r["resample_fn"])
    stage2_results[label] = {"mean": aucs2.mean(), "std": aucs2.std(), "bagged": bagged_auc2}
    print(f"{label:<28} | {aucs2.mean():>10.5f} | {aucs2.std():>8.5f} | {bagged_auc2:>13.5f}")

print()
print("비교 기준:")
print("  기존 단일시드(1004) binary baseline: 0.73998")
print("  10시드 배깅 baseline: 0.74036 (기존 파라미터) / 0.74037 (견고탐색 파라미터)")
print("  노이즈 표준편차(참고): 0.00011 (baseline 10시드 기준 실측값)")


Stage 2 정밀검증 후보 (baseline보다 +0.0002 이상): 없음

설정                           |    10시드 평균 |     표준편차 |    10시드 배깅AUC
----------------------------------------------------------------------
baseline                     |    0.73997 |  0.00013 |       0.74037

비교 기준:
  기존 단일시드(1004) binary baseline: 0.73998
  10시드 배깅 baseline: 0.74036 (기존 파라미터) / 0.74037 (견고탐색 파라미터)
  노이즈 표준편차(참고): 0.00011 (baseline 10시드 기준 실측값)


## 6. 결론 가이드

- Stage 2의 `baseline` 행이 기존 벤치마크(0.74036~0.74037)와 비슷하게 나오면, 이 노트북의 파이프라인이
  기존 결과와 잘 맞는다는 검증이 된 것 (재현성 확인)
- 후보 설정 중 **10시드 배깅 AUC가 baseline 배깅 AUC보다 노이즈(0.00011)의 2~3배 이상 높으면** 진짜
  개선으로 볼 수 있음 — 그 설정의 OOF/모델을 블렌딩 후보로 추가 검토
- 전부 baseline과 비슷하거나 낮다면, "클래스 불균형 처리 전체 계열(가중치/오버샘플링/언더샘플링)이
  이 ROC-AUC 과제에는 효과가 없다"는 훨씬 강한 결론을 내릴 수 있음 (예전엔 2가지 기법만, 단일 시드로
  봤지만 이번엔 6가지 기법을 멀티시드로 봤으니까)
